# 2.0 DeepLog Model Prototyping & Live Trace Simulation
**Objective:** Rapid prototype iteration, Top-K parameter tuning ($k \in [1, 3, 5, 9, 15]$), and live step-by-step stream testing.

In [ ]:
# ============================================================
# STEP 1: Imports & Model Loading
# ============================================================
import os, sys, pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

sys.path.append('..')
from src.config import WINDOW_H, TOP_K, ANOMALY_THRESHOLD
from src.modeling.predict import predict_topk_flags

model = keras.models.load_model('../models/lstm_log_anomaly_model.keras')
data = np.load('../data/processed/train_val_test.npz')
X_test, y_test, sid_test = data['X_test'], data['y_test'], data['sid_test']
print("Model and test windows loaded successfully.")

In [ ]:
# ============================================================
# STEP 2: Live Trace Simulation (Step-by-Step Inference)
# ============================================================
def simulate_live_log_stream(sample_session, history_window=10, k=9):
    """
    Simulates real-time log stream processing item-by-item.
    """
    print(f"\n--- SIMULATING LIVE STREAM INGESTION (Session Length = {len(sample_session)}) ---")
    history = list(sample_session[:history_window])
    
    for t in range(history_window, len(sample_session)):
        incoming_event = sample_session[t]
        input_seq = np.array([history[-history_window:]]) # shape: (1, h)
        
        # Model predicts distribution
        probs = model.predict(input_seq, verbose=0)[0]
        top_k_predicted_events = np.argpartition(probs, -k)[-k:]
        
        is_anomaly = incoming_event not in top_k_predicted_events
        status = "🔴 ANOMALY DETECTED" if is_anomaly else "🟢 NORMAL"
        
        print(f"Step {t:02d} | Input Context: {history[-5:]}... | Actual Event: {incoming_event:02d} | Top-K Probs: {top_k_predicted_events} | Status: {status}")
        history.append(incoming_event)

# Test live stream on one test session
sample_idx = 0
sample_session_ids = np.unique(sid_test)
sample_window_mask = sid_test == sample_session_ids[sample_idx]
sample_events = list(X_test[sample_window_mask][0]) + list(y_test[sample_window_mask])

simulate_live_log_stream(sample_events, history_window=WINDOW_H, k=TOP_K)